In [1]:
# Day 6: Advanced Visualizations (Part 2)
# Goal: Create advanced charts and extract insights

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Set style
sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (12, 6)

print("="*60)
print("DAY 6: ADVANCED VISUALIZATIONS & INSIGHTS")
print("="*60)

# ============================================================
# VISUALIZATION 9: Resolution Status Breakdown (Pie Chart)
# ============================================================
print("\n📊 Creating Chart 9: Resolution Status Distribution")

df9 = pd.read_csv('sql_results/q9_resolution_status.csv')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart
colors = ['#06A77D', '#F77F00', '#E63946', '#2E86AB']
wedges, texts, autotexts = ax1.pie(df9['total_complaints'], labels=df9['resolution_status'],
                                     autopct='%1.1f%%', startangle=90, colors=colors,
                                     textprops={'fontsize': 11})
ax1.set_title('Resolution Status Distribution', fontsize=14, fontweight='bold')

# Make percentage text bold
for autotext in autotexts:
    autotext.set_color('white')
    autotext.set_fontweight('bold')

# Bar chart with satisfaction
ax2.bar(df9['resolution_status'], df9['avg_satisfaction'], color=colors)
ax2.set_title('Avg Satisfaction by Resolution Status', fontsize=14, fontweight='bold')
ax2.set_ylabel('Satisfaction Score (1-5)', fontsize=11)
ax2.set_ylim(0, 5)
ax2.tick_params(axis='x', rotation=45)

# Add value labels
for i, v in enumerate(df9['avg_satisfaction']):
    if pd.notna(v):
        ax2.text(i, v + 0.1, f'{v:.2f}', ha='center', va='bottom', fontsize=10)

plt.tight_layout()
plt.savefig('visualizations/09_resolution_status.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: visualizations/09_resolution_status.png")

# ============================================================
# VISUALIZATION 10: Repeat vs First-Time Complaints
# ============================================================
print("\n📊 Creating Chart 10: Repeat vs First-Time Comparison")

df8 = pd.read_csv('sql_results/q8_repeat_complaints.csv')

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Chart 1: Total complaints
axes[0, 0].bar(df8['is_repeat_complaint'], df8['total_complaints'], color=['#2E86AB', '#E63946'])
axes[0, 0].set_title('Total Complaints', fontsize=12, fontweight='bold')
axes[0, 0].set_ylabel('Count', fontsize=10)
for i, v in enumerate(df8['total_complaints']):
    axes[0, 0].text(i, v + 100, str(v), ha='center', va='bottom', fontsize=10)

# Chart 2: Resolution time
axes[0, 1].bar(df8['is_repeat_complaint'], df8['avg_resolution_hours'], color=['#06A77D', '#F77F00'])
axes[0, 1].set_title('Avg Resolution Time', fontsize=12, fontweight='bold')
axes[0, 1].set_ylabel('Hours', fontsize=10)
for i, v in enumerate(df8['avg_resolution_hours']):
    axes[0, 1].text(i, v + 1, f'{v:.1f}h', ha='center', va='bottom', fontsize=10)

# Chart 3: Satisfaction
axes[1, 0].bar(df8['is_repeat_complaint'], df8['avg_satisfaction'], color=['#2E86AB', '#E63946'])
axes[1, 0].set_title('Avg Satisfaction Score', fontsize=12, fontweight='bold')
axes[1, 0].set_ylabel('Score (1-5)', fontsize=10)
axes[1, 0].set_ylim(0, 5)
for i, v in enumerate(df8['avg_satisfaction']):
    if pd.notna(v):
        axes[1, 0].text(i, v + 0.1, f'{v:.2f}', ha='center', va='bottom', fontsize=10)

# Chart 4: Summary text
axes[1, 1].axis('off')
summary_text = f"""
KEY FINDINGS:

• First-Time Complaints: {df8[df8['is_repeat_complaint']=='First-Time']['total_complaints'].values[0]:,}
• Repeat Complaints: {df8[df8['is_repeat_complaint']=='Repeat']['total_complaints'].values[0]:,}

• Repeat complaints show customers 
  are not satisfied with initial 
  resolution

• Resolution time difference: 
  {abs(df8['avg_resolution_hours'].iloc[0] - df8['avg_resolution_hours'].iloc[1]):.1f} hours

• Satisfaction gap: 
  {abs(df8['avg_satisfaction'].iloc[0] - df8['avg_satisfaction'].iloc[1]):.2f} points
"""
axes[1, 1].text(0.1, 0.5, summary_text, fontsize=11, verticalalignment='center',
                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.3))

plt.suptitle('First-Time vs Repeat Complaints Analysis', fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.savefig('visualizations/10_repeat_analysis.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: visualizations/10_repeat_analysis.png")

# ============================================================
# VISUALIZATION 11: Holiday Season Impact
# ============================================================
print("\n📊 Creating Chart 11: Holiday Season Impact")

df11 = pd.read_csv('sql_results/q11_holiday_impact.csv')

# Aggregate by season and category
holiday_summary = df11.groupby(['is_holiday_season', 'complaint_category']).agg({
    'total_complaints': 'sum'
}).reset_index()

# Pivot for grouped bar chart
pivot_data = holiday_summary.pivot(index='complaint_category', 
                                    columns='is_holiday_season', 
                                    values='total_complaints').fillna(0)

fig, ax = plt.subplots(figsize=(14, 7))
pivot_data.plot(kind='bar', ax=ax, color=['#2E86AB', '#E63946'], width=0.8)

ax.set_title('Complaint Categories: Holiday Season vs Regular Season', fontsize=16, fontweight='bold', pad=20)
ax.set_xlabel('Complaint Category', fontsize=12)
ax.set_ylabel('Number of Complaints', fontsize=12)
ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
ax.legend(title='Season', fontsize=11)

# Add value labels on bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.0f', padding=3, fontsize=9)

plt.tight_layout()
plt.savefig('visualizations/11_holiday_impact.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: visualizations/11_holiday_impact.png")

# ============================================================
# VISUALIZATION 12: Priority vs Satisfaction Heatmap
# ============================================================
print("\n📊 Creating Chart 12: Priority vs Satisfaction Analysis")

df12 = pd.read_csv('sql_results/q12_priority_satisfaction.csv')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Bar chart: Priority levels
priority_order = ['critical', 'high', 'medium', 'low']
df12['priority_level'] = pd.Categorical(df12['priority_level'], categories=priority_order, ordered=True)
df12 = df12.sort_values('priority_level')

colors_priority = ['#8B0000', '#E63946', '#F77F00', '#06A77D']
ax1.barh(df12['priority_level'], df12['total_complaints'], color=colors_priority)
ax1.set_title('Complaints by Priority Level', fontsize=14, fontweight='bold')
ax1.set_xlabel('Number of Complaints', fontsize=11)
ax1.set_ylabel('Priority', fontsize=11)

for i, v in enumerate(df12['total_complaints']):
    ax1.text(v + 50, i, str(v), va='center', fontsize=10)

# Scatter: Resolution time vs Satisfaction by priority
for priority, color in zip(priority_order, colors_priority):
    data = df12[df12['priority_level'] == priority]
    ax2.scatter(data['avg_resolution_hours'], data['avg_satisfaction'], 
               s=data['total_complaints']/5, alpha=0.7, c=color, label=priority,
               edgecolors='black', linewidth=1)

ax2.set_title('Resolution Time vs Satisfaction by Priority', fontsize=14, fontweight='bold')
ax2.set_xlabel('Avg Resolution Time (hours)', fontsize=11)
ax2.set_ylabel('Avg Satisfaction Score', fontsize=11)
ax2.legend(title='Priority', fontsize=10)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('visualizations/12_priority_analysis.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: visualizations/12_priority_analysis.png")

# ============================================================
# VISUALIZATION 13: Executive Summary Dashboard
# ============================================================
print("\n📊 Creating Chart 13: Executive Summary Dashboard")

# Load main dataset for KPIs
df_main = pd.read_csv('customer_complaints_enriched.csv')

fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(3, 3, hspace=0.4, wspace=0.3)

# KPI Cards
ax_kpi1 = fig.add_subplot(gs[0, 0])
ax_kpi2 = fig.add_subplot(gs[0, 1])
ax_kpi3 = fig.add_subplot(gs[0, 2])

# KPI 1: Total Complaints
ax_kpi1.axis('off')
total_complaints = len(df_main)
ax_kpi1.text(0.5, 0.6, f'{total_complaints:,}', ha='center', va='center', 
            fontsize=40, fontweight='bold', color='#2E86AB')
ax_kpi1.text(0.5, 0.3, 'Total Complaints', ha='center', va='center', fontsize=14)
ax_kpi1.add_patch(plt.Rectangle((0.1, 0.1), 0.8, 0.8, fill=False, edgecolor='#2E86AB', linewidth=3))

# KPI 2: Avg Resolution Time
ax_kpi2.axis('off')
avg_resolution = df_main['resolution_time_hours'].mean()
ax_kpi2.text(0.5, 0.6, f'{avg_resolution:.1f}h', ha='center', va='center', 
            fontsize=40, fontweight='bold', color='#F77F00')
ax_kpi2.text(0.5, 0.3, 'Avg Resolution Time', ha='center', va='center', fontsize=14)
ax_kpi2.add_patch(plt.Rectangle((0.1, 0.1), 0.8, 0.8, fill=False, edgecolor='#F77F00', linewidth=3))

# KPI 3: Avg Satisfaction
ax_kpi3.axis('off')
avg_satisfaction = df_main['satisfaction_score'].mean()
ax_kpi3.text(0.5, 0.6, f'{avg_satisfaction:.2f}/5', ha='center', va='center', 
            fontsize=40, fontweight='bold', color='#06A77D')
ax_kpi3.text(0.5, 0.3, 'Avg Satisfaction', ha='center', va='center', fontsize=14)
ax_kpi3.add_patch(plt.Rectangle((0.1, 0.1), 0.8, 0.8, fill=False, edgecolor='#06A77D', linewidth=3))

# Chart 1: Monthly Trend (small version)
ax1 = fig.add_subplot(gs[1, :])
df1 = pd.read_csv('sql_results/q1_monthly_trends.csv')
df1['date'] = pd.to_datetime(df1['year'].astype(str) + '-' + df1['month'].astype(str) + '-01')
ax1.plot(df1['date'], df1['total_complaints'], marker='o', linewidth=2, color='#2E86AB')
ax1.set_title('Complaint Trend Over Time', fontsize=12, fontweight='bold')
ax1.set_xlabel('Month', fontsize=10)
ax1.set_ylabel('Complaints', fontsize=10)
ax1.tick_params(axis='x', rotation=45, labelsize=8)
ax1.grid(True, alpha=0.3)

# Chart 2: Top Categories
ax2 = fig.add_subplot(gs[2, 0])
df2 = pd.read_csv('sql_results/q2_complaint_categories.csv').head(5)
ax2.barh(df2['complaint_category'], df2['total_complaints'], color='#E63946')
ax2.set_title('Top 5 Complaint Types', fontsize=11, fontweight='bold')
ax2.set_xlabel('Count', fontsize=9)
ax2.tick_params(labelsize=8)

# Chart 3: Resolution Status
ax3 = fig.add_subplot(gs[2, 1])
df9 = pd.read_csv('sql_results/q9_resolution_status.csv')
colors = ['#06A77D', '#F77F00', '#E63946', '#2E86AB']
ax3.pie(df9['total_complaints'], labels=df9['resolution_status'], autopct='%1.0f%%',
        colors=colors, textprops={'fontsize': 8})
ax3.set_title('Resolution Status', fontsize=11, fontweight='bold')

# Chart 4: Key Insights Text
ax4 = fig.add_subplot(gs[2, 2])
ax4.axis('off')

# Calculate insights
resolved_pct = (df9[df9['resolution_status']=='resolved']['total_complaints'].sum() / 
                df9['total_complaints'].sum() * 100)
high_value_count = df_main['is_high_value_customer'].sum()

insights = f"""
KEY INSIGHTS:

✓ {resolved_pct:.1f}% resolution rate

✓ {high_value_count:,} high-value 
  customers affected

✓ Average {avg_resolution:.0f} hour 
  resolution time

✓ Top issue: 
  {df2.iloc[0]['complaint_category']}

⚠ Focus areas:
  • Product quality
  • Delivery times
  • Store training
"""
ax4.text(0.1, 0.5, insights, fontsize=10, verticalalignment='center',
         bbox=dict(boxstyle='round', facecolor='#FFF9E6', alpha=0.8))

plt.suptitle('CUSTOMER COMPLAINT ANALYSIS - EXECUTIVE SUMMARY', 
             fontsize=18, fontweight='bold', y=0.98)
plt.savefig('visualizations/13_executive_dashboard.png', dpi=300, bbox_inches='tight')
plt.close()
print("✅ Saved: visualizations/13_executive_dashboard.png")

print("\n" + "="*60)
print("✅ DAY 6 COMPLETE - ALL VISUALIZATIONS CREATED!")
print("="*60)
print("\nTotal visualizations created: 13")
print("\nAll charts saved in 'visualizations/' folder")


DAY 6: ADVANCED VISUALIZATIONS & INSIGHTS

📊 Creating Chart 9: Resolution Status Distribution
✅ Saved: visualizations/09_resolution_status.png

📊 Creating Chart 10: Repeat vs First-Time Comparison
✅ Saved: visualizations/10_repeat_analysis.png

📊 Creating Chart 11: Holiday Season Impact
✅ Saved: visualizations/11_holiday_impact.png

📊 Creating Chart 12: Priority vs Satisfaction Analysis
✅ Saved: visualizations/12_priority_analysis.png

📊 Creating Chart 13: Executive Summary Dashboard


C:\Users\deeks\AppData\Local\Temp\ipykernel_14212\2199666796.py:290: UserWarning: Glyph 10003 (\N{CHECK MARK}) missing from font(s) Arial.
  plt.savefig('visualizations/13_executive_dashboard.png', dpi=300, bbox_inches='tight')
C:\Users\deeks\AppData\Local\Temp\ipykernel_14212\2199666796.py:290: UserWarning: Glyph 9888 (\N{WARNING SIGN}) missing from font(s) Arial.
  plt.savefig('visualizations/13_executive_dashboard.png', dpi=300, bbox_inches='tight')


✅ Saved: visualizations/13_executive_dashboard.png

✅ DAY 6 COMPLETE - ALL VISUALIZATIONS CREATED!

Total visualizations created: 13

All charts saved in 'visualizations/' folder
